# Задание 1. Складской учёт


In [6]:
# ===== Задание 1. Складской учёт =====

class WarehouseError(Exception):
    """Базовый класс ошибок складского учёта."""
    def __init__(self, message, line_no=None):
        self.line_no = line_no
        super().__init__(message)


class UnknownItemError(WarehouseError):
    """Товар отсутствует в справочнике склада."""
    def __init__(self, item, line_no=None):
        self.item = item
        super().__init__(f"товар «{item}» не найден", line_no)


class InvalidQuantityError(WarehouseError):
    """Количество не является положительным целым числом."""
    def __init__(self, item, value, line_no=None):
        self.item = item
        self.value = value
        super().__init__(
            f"«{item}»: количество «{value}» должно быть положительным",
            line_no)


class InsufficientStockError(WarehouseError):
    """На складе недостаточно товара для выдачи."""
    def __init__(self, item, requested, available, line_no=None):
        self.item = item
        self.requested = requested
        self.available = available
        super().__init__(
            f"«{item}»: запрошено {requested}, в наличии {available}",
            line_no)


class OperationFormatError(WarehouseError):
    """Строка журнала операций имеет неверный формат."""


# ---------- функции нижнего уровня ----------

def receive(stock, item, qty):
    """Приход товара: добавляет количество к остатку."""
    stock[item] = stock.get(item, 0) + qty


def issue(stock, item, qty):
    """Расход товара; изменяет словарь только при успешной проверке.

    Raises:
        UnknownItemError: товара нет в справочнике.
        InsufficientStockError: товара недостаточно.
    """
    if item not in stock:
        raise UnknownItemError(item)
    available = stock[item]
    if qty > available:
        raise InsufficientStockError(item, qty, available)
    stock[item] = available - qty


def parse_operation(line, line_no):
    """Разбирает строку журнала → (операция, товар, количество).

    Raises:
        OperationFormatError: неверное число полей или неизвестный тип.
        InvalidQuantityError: количество не положительное целое.
    """
    parts = [p.strip() for p in line.rstrip("\n").split(";")]
    if len(parts) != 3:
        raise OperationFormatError(f"полей {len(parts)} вместо 3", line_no)
    op, item, raw_qty = parts
    if op not in ("приход", "расход"):
        raise OperationFormatError(f"неизвестная операция «{op}»", line_no)
    try:
        qty = int(raw_qty)
    except ValueError as e:
        raise InvalidQuantityError(item, raw_qty, line_no) from e
    if qty <= 0:
        raise InvalidQuantityError(item, raw_qty, line_no)
    return op, item, qty


def process(name, stock):
    """Обрабатывает журнал, накапливает объекты исключений.

    Returns:
        Список WarehouseError в порядке появления.
    """
    errors = []
    with open(name, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                op, item, qty = parse_operation(line, line_no)
                if op == "приход":
                    receive(stock, item, qty)
                else:
                    issue(stock, item, qty)
            except WarehouseError as e:
                if e.line_no is None:
                    e.line_no = line_no
                errors.append(e)
    return errors


# ---------- подготовка файла и вывод ----------

def make_operations_file(name):
    data = (
        "приход; болт М6;100\n"
        "приход; гайка М8;50\n"
        "приход; винт М4;200\n"
        "расход; шайба М10;5\n"
        "расход; болт М6;30\n"
        "расход; гайка М8;80\n"
        "расход; винт М4;215\n"
        "расход; болт М6;- 5\n"
        "списание; гайка М8;10\n"
        "приход; болт\n"
        "приход; гайка М8;abc\n"
        "приход; винт М4;-10\n"
    )
    with open(name, "w", encoding="utf-8") as f:
        f.write(data)


def main():
    filename = "operations.txt"
    make_operations_file(filename)
    stock = {}
    errors = process(filename, stock)

    for e in errors:
        print(f"строка {e.line_no}: {type(e).__name__} {e}")
    print("Остатки:", stock)

    need = {}
    for e in errors:
        if isinstance(e, InsufficientStockError):
            need[e.item] = need.get(e.item, 0) + e.requested - e.available
    print("Заявка поставщику:", need)


main()

строка 4: UnknownItemError товар «шайба М10» не найден
строка 6: InsufficientStockError «гайка М8»: запрошено 80, в наличии 50
строка 7: InsufficientStockError «винт М4»: запрошено 215, в наличии 200
строка 8: InvalidQuantityError «болт М6»: количество «- 5» должно быть положительным
строка 9: OperationFormatError неизвестная операция «списание»
строка 10: OperationFormatError полей 2 вместо 3
строка 11: InvalidQuantityError «гайка М8»: количество «abc» должно быть положительным
строка 12: InvalidQuantityError «винт М4»: количество «-10» должно быть положительным
Остатки: {'болт М6': 70, 'гайка М8': 50, 'винт М4': 200}
Заявка поставщику: {'гайка М8': 30, 'винт М4': 15}


# Задание 2. Проверка регистрационных данных


In [7]:
# ===== Задание 2. Валидатор анкет =====
from collections import Counter


class ValidationError(Exception):
    """Базовый класс ошибок валидации анкеты."""
    def __init__(self, field, value, message):
        self.field = field
        self.value = value
        super().__init__(message)


class EmailError(ValidationError):
    """Ошибка в поле e-mail."""
    def __init__(self, value, reason):
        super().__init__("email", value, f"поле «email» = {value!r}: {reason}")


class PhoneError(ValidationError):
    """Ошибка в поле телефона."""
    def __init__(self, value, reason):
        super().__init__("телефон", value, f"поле «телефон» = {value!r}: {reason}")


class AgeError(ValidationError):
    """Ошибка в поле возраста."""
    def __init__(self, value, reason):
        super().__init__("возраст", value, f"поле «возраст» = {value!r}: {reason}")


class RequiredFieldError(ValidationError):
    """Обязательное поле не заполнено."""
    def __init__(self, field):
        super().__init__(field, "", f"поле «{field}» не заполнено")


class FormError(Exception):
    """Анкета содержит одну или несколько ошибок."""
    def __init__(self, errors):
        self.errors = errors
        super().__init__(f"в анкете {len(errors)} ошибок")


# ---------- проверки полей ----------

def check_email(value):
    """Проверяет e-mail. Возвращает нормализованное значение.

    Raises:
        EmailError: если строка не является корректным e-mail.
    """
    if value.count("@") != 1:
        raise EmailError(value, "должен быть ровно один символ @")
    local, domain = value.split("@")
    if not local:
        raise EmailError(value, "пустая часть до @")
    if not domain:
        raise EmailError(value, "пустая часть после @")
    if "." not in domain:
        raise EmailError(value, "нет точки в имени домена")
    return value.lower()


def check_phone(value):
    """Проверяет телефон. Возвращает номер в формате +7XXXXXXXXXX.

    Raises:
        PhoneError: если телефон не соответствует формату.
    """
    digits = "".join(ch for ch in value if ch.isdigit() or ch == "+")
    if digits.startswith("+7") and len(digits) == 12:
        return "+7" + digits[2:]
    if digits.startswith("8") and len(digits) == 11:
        return "+7" + digits[1:]
    raise PhoneError(value, "ожидается +7 и 10 цифр или 8 и 10 цифр")


def check_age(value):
    """Проверяет возраст. Возвращает целое число.

    Raises:
        AgeError: если возраст не целое от 16 до 100.
    """
    try:
        age = int(value)
    except (ValueError, TypeError) as e:
        raise AgeError(value, "должно быть целым числом") from e
    if not 16 <= age <= 100:
        raise AgeError(value, "допустимо от 16 до 100")
    return age


def validate_form(form):
    """Проверяет все поля анкеты, собирая ошибки.

    Returns:
        Нормализованная анкета.

    Raises:
        FormError: если хотя бы одно поле содержит ошибку.
    """
    errors = []
    normalized = {}

    fio = form.get("ФИО", "").strip()
    if not fio:
        errors.append(RequiredFieldError("ФИО"))
    else:
        normalized["ФИО"] = fio

    email = form.get("email", "").strip()
    if not email:
        errors.append(RequiredFieldError("email"))
    else:
        try:
            normalized["email"] = check_email(email)
        except ValidationError as e:
            errors.append(e)

    phone = form.get("телефон", "").strip()
    if not phone:
        errors.append(RequiredFieldError("телефон"))
    else:
        try:
            normalized["телефон"] = check_phone(phone)
        except ValidationError as e:
            errors.append(e)

    age = form.get("возраст", "").strip()
    if not age:
        errors.append(RequiredFieldError("возраст"))
    else:
        try:
            normalized["возраст"] = check_age(age)
        except ValidationError as e:
            errors.append(e)

    if errors:
        raise FormError(errors)
    return normalized


# ---------- ввод-вывод ----------

def parse_form_line(line):
    parts = [p.strip() for p in line.rstrip("\n").split(";")]
    while len(parts) < 4:
        parts.append("")
    return {"ФИО": parts[0], "email": parts[1],
            "телефон": parts[2], "возраст": parts[3]}


def plural_errors(n):
    if n % 10 == 1 and n % 100 != 11:
        return f"{n} ошибка"
    if 2 <= n % 10 <= 4 and not (12 <= n % 100 <= 14):
        return f"{n} ошибки"
    return f"{n} ошибок"


def make_forms_file(name):
    data = (
        "Иванов Иван;ivanov@mail.ru;+7 916 123-45-67;30\n"
        "Петров Пётр;petrov@mail.ru;89161234567;25\n"
        "Сидоров Сидор;sidorov@mail.ru;+7(916)123-45-67;40\n"
        "Кузнецов Кузьма;kuznecov@mail.ru;8-916-123-45-67;35\n"
        "Петрова Анна;petrova@mail;+79161234567;20\n"
        "Смирнова Ольга;smirnova@mail.ru;12345;35\n"
        "Попов Павел;popov@mail.ru;8-916;150\n"
        ";test@mail.ru;abc;20\n"
    )
    with open(name, "w", encoding="utf-8") as f:
        f.write(data)


def main():
    filename = "forms.txt"
    make_forms_file(filename)
    accepted, total = 0, 0
    field_errors = Counter()

    with open(filename, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            if not line.strip():
                continue
            total += 1
            form = parse_form_line(line)
            try:
                norm = validate_form(form)
            except FormError as e:
                fio = form["ФИО"] or "–"
                print(f"Анкета {i} ({fio}): {plural_errors(len(e.errors))}")
                for err in e.errors:
                    print(f"  {type(err).__name__}: {err}")
                    field_errors[err.field] += 1
            else:
                accepted += 1
                print(f"Анкета {i} ({norm['ФИО']}): принята")

    if field_errors:
        field, count = field_errors.most_common(1)[0]
        print(f"Принято {accepted} из {total}; "
              f"чаще всего ошибки в поле «{field}» ({count})")
    else:
        print(f"Принято {accepted} из {total}; ошибок нет")


main()

Анкета 1 (Иванов Иван): принята
Анкета 2 (Петров Пётр): принята
Анкета 3 (Сидоров Сидор): принята
Анкета 4 (Кузнецов Кузьма): принята
Анкета 5 (Петрова Анна): 1 ошибка
  EmailError: поле «email» = 'petrova@mail': нет точки в имени домена
Анкета 6 (Смирнова Ольга): 1 ошибка
  PhoneError: поле «телефон» = '12345': ожидается +7 и 10 цифр или 8 и 10 цифр
Анкета 7 (Попов Павел): 2 ошибки
  PhoneError: поле «телефон» = '8-916': ожидается +7 и 10 цифр или 8 и 10 цифр
  AgeError: поле «возраст» = '150': допустимо от 16 до 100
Анкета 8 (–): 2 ошибки
  RequiredFieldError: поле «ФИО» не заполнено
  PhoneError: поле «телефон» = 'abc': ожидается +7 и 10 цифр или 8 и 10 цифр
Принято 4 из 8; чаще всего ошибки в поле «телефон» (3)


# Задание 3. Загрузка файла настроек


In [8]:
# ===== Задание 3. Загрузчик INI-файла =====

class ConfigError(Exception):
    """Базовый класс ошибок загрузки конфигурации."""


class ConfigSyntaxError(ConfigError):
    """Синтаксическая ошибка в файле конфигурации."""
    def __init__(self, line_no, line, reason):
        self.line_no = line_no
        self.line = line
        super().__init__(f"строка {line_no}: «{line.strip()}» – {reason}")


class MissingKeyError(ConfigError):
    """Отсутствует обязательный ключ."""
    def __init__(self, section, key):
        self.section = section
        self.key = key
        super().__init__(f"в секции [{section}] нет ключа «{key}»")


class ConfigValueError(ConfigError):
    """Значение ключа не соответствует ожиданиям."""
    def __init__(self, section, key, value, expected):
        self.section = section
        self.key = key
        self.value = value
        self.expected = expected
        super().__init__(f"[{section}] {key} = {value!r}: ожидается {expected}")


class DuplicateKeyError(ConfigError):
    """Ключ уже определён в этой секции."""
    def __init__(self, line_no, section, key):
        self.line_no = line_no
        self.section = section
        self.key = key
        super().__init__(
            f"строка {line_no}: ключ «{key}» в секции [{section}] уже задан")


# ---------- разбор ----------

def parse_config(name):
    """Разбирает INI-файл → {section: {key: value}}.

    Raises:
        ConfigSyntaxError: неверная структура файла.
        DuplicateKeyError: повторный ключ в секции.
    """
    config = {}
    current = None
    with open(name, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            if stripped.startswith("["):
                if not stripped.endswith("]"):
                    raise ConfigSyntaxError(line_no, line, "незакрытая скобка")
                section = stripped[1:-1].strip()
                if not section:
                    raise ConfigSyntaxError(line_no, line, "пустое имя секции")
                config.setdefault(section, {})
                current = section
                continue
            if current is None:
                raise ConfigSyntaxError(line_no, line, "строка вне секции")
            if "=" not in stripped:
                raise ConfigSyntaxError(line_no, line, "нет знака «=»")
            key, value = stripped.split("=", 1)
            key = key.strip()
            value = value.strip()
            if key in config[current]:
                raise DuplicateKeyError(line_no, current, key)
            config[current][key] = value
    return config


# ---------- извлечение значений ----------

def get_int(cfg, section, key, low, high):
    """Извлекает целое из диапазона [low, high].

    Raises:
        MissingKeyError: ключ отсутствует.
        ConfigValueError: не целое или вне диапазона.
    """
    if section not in cfg or key not in cfg[section]:
        raise MissingKeyError(section, key)
    raw = cfg[section][key]
    try:
        value = int(raw)
    except ValueError as e:
        raise ConfigValueError(section, key, raw,
                               f"целое {low}..{high}") from e
    if not low <= value <= high:
        raise ConfigValueError(section, key, raw, f"целое {low}..{high}")
    return value


def get_bool(cfg, section, key):
    """Извлекает логическое значение (yes/no/true/false/1/0).

    Raises:
        MissingKeyError: ключ отсутствует.
        ConfigValueError: значение не из допустимого набора.
    """
    if section not in cfg or key not in cfg[section]:
        raise MissingKeyError(section, key)
    raw = cfg[section][key].strip().lower()
    if raw in ("yes", "true", "1"):
        return True
    if raw in ("no", "false", "0"):
        return False
    raise ConfigValueError(section, key, cfg[section][key],
                           "одно из yes/no/true/false/1/0")


def get_str(cfg, section, key, choices=None):
    """Извлекает строку с проверкой допустимых вариантов.

    Raises:
        MissingKeyError: ключ отсутствует.
        ConfigValueError: значение не входит в choices.
    """
    if section not in cfg or key not in cfg[section]:
        raise MissingKeyError(section, key)
    value = cfg[section][key]
    if choices is not None and value not in choices:
        raise ConfigValueError(section, key, value, f"одно из {choices}")
    return value


def load_settings(name):
    """Загружает итоговые настройки приложения."""
    cfg = parse_config(name)
    return {
        "host": get_str(cfg, "database", "host"),
        "database": get_str(cfg, "database", "name"),
        "debug": get_bool(cfg, "app", "debug"),
        "log_level": get_str(cfg, "app", "log_level",
                             choices=("debug", "info", "warning", "error")),
        "port": get_int(cfg, "server", "port", 1, 65535),
        "timeout": get_int(cfg, "server", "timeout", 1, 3600),
        "workers": get_int(cfg, "server", "workers", 1, 64),
    }


# ---------- создание файлов и демонстрация ----------

def make_files():
    with open("settings_ok.ini", "w", encoding="utf-8") as f:
        f.write(
            "# приложение\n"
            "[database]\n"
            "host = localhost\n"
            "name = meteo\n"
            "\n"
            "[app]\n"
            "debug = no\n"
            "log_level = info\n"
            "\n"
            "[server]\n"
            "port = 5432\n"
            "timeout = 30\n"
            "workers = 4\n"
        )
    with open("settings_1.ini", "w", encoding="utf-8") as f:
        f.write(
            "# приложение\n"
            "[database]\n"
            "host = localhost\n"
            "port 5432\n"
            "name = meteo\n"
        )
    with open("settings_2.ini", "w", encoding="utf-8") as f:
        f.write(
            "# приложение\n"
            "[database]\n"
            "port = 5432\n"
            "name = meteo\n"
            "\n"
            "[app]\n"
            "debug = no\n"
            "log_level = info\n"
            "\n"
            "[server]\n"
            "port = 5432\n"
            "timeout = 30\n"
            "workers = 4\n"
        )
    with open("settings_3.ini", "w", encoding="utf-8") as f:
        f.write(
            "# приложение\n"
            "[database]\n"
            "host = localhost\n"
            "name = meteo\n"
            "\n"
            "[app]\n"
            "debug = no\n"
            "log_level = info\n"
            "\n"
            "[server]\n"
            "port = восемьдесят\n"
            "timeout = 30\n"
            "workers = 4\n"
        )
    with open("settings_4.ini", "w", encoding="utf-8") as f:
        f.write(
            "# приложение\n"
            "[database]\n"
            "host = localhost\n"
            "name = meteo\n"
            "\n"
            "[app]\n"
            "debug = no\n"
            "log_level = info\n"
            "debug = yes\n"
        )


def main():
    make_files()
    for name in ("settings_ok.ini", "settings_1.ini", "settings_2.ini",
                 "settings_3.ini", "settings_4.ini"):
        try:
            settings = load_settings(name)
        except ConfigError as e:
            print(f"{name}: {type(e).__name__}: {e}")
        else:
            print(f"{name}: {settings}")


main()

settings_ok.ini: {'host': 'localhost', 'database': 'meteo', 'debug': False, 'log_level': 'info', 'port': 5432, 'timeout': 30, 'workers': 4}
settings_1.ini: ConfigSyntaxError: строка 4: «port 5432» – нет знака «=»
settings_2.ini: MissingKeyError: в секции [database] нет ключа «host»
settings_3.ini: ConfigValueError: [server] port = 'восемьдесят': ожидается целое 1..65535
settings_4.ini: DuplicateKeyError: строка 9: ключ «debug» в секции [app] уже задан


# Задание 4. Бронирование аудиторий

In [4]:
# ===== Задание 4. Бронирование аудиторий =====
from collections import defaultdict


class BookingError(Exception):
    """Базовый класс ошибок бронирования."""


class RoomNotFoundError(BookingError):
    """Аудитория отсутствует в справочнике."""
    def __init__(self, room):
        self.room = room
        super().__init__(f"аудитории {room} нет в справочнике")


class InvalidTimeError(BookingError):
    """Некорректное время бронирования."""
    def __init__(self, start, end, reason):
        self.start = start
        self.end = end
        self.reason = reason
        super().__init__(reason)


class CapacityError(BookingError):
    """Число участников превышает вместимость аудитории."""
    def __init__(self, room, capacity, people):
        self.room = room
        self.capacity = capacity
        self.people = people
        super().__init__(
            f"аудитория {room}: вместимость {capacity}, заявлено {people}")


class TimeConflictError(BookingError):
    """Пересечение с существующей бронью."""
    def __init__(self, room, requested, existing):
        self.room = room
        self.requested = requested
        self.existing = existing
        who, start, end = existing[4], existing[2], existing[3]
        super().__init__(f"занято группой {who} ({start}-{end})")


# ---------- работа со временем ----------

DAY_ORDER = {"пн": 1, "вт": 2, "ср": 3, "чт": 4, "пт": 5, "сб": 6, "вс": 7}


def parse_time(text):
    """'чч:мм' → минуты от начала суток.

    Raises:
        ValueError: неверный формат или время вне суток.
    """
    h, m = text.strip().split(":")
    h, m = int(h), int(m)
    if not 0 <= h < 24 or not 0 <= m < 60:
        raise ValueError("время вне суток")
    return h * 60 + m


def check_time(start, end):
    """Проверяет интервал времени.

    Raises:
        InvalidTimeError: неверный формат, порядок или границы.
    """
    try:
        s = parse_time(start)
    except (ValueError, AttributeError) as e:
        raise InvalidTimeError(start, end, "неверный формат времени") from e
    try:
        e_min = parse_time(end)
    except (ValueError, AttributeError) as e:
        raise InvalidTimeError(start, end, "неверный формат времени") from e
    if s >= e_min:
        raise InvalidTimeError(start, end, "начало позже окончания")
    if s < 8 * 60 or e_min > 21 * 60:
        raise InvalidTimeError(start, end, "вне пределов 08:00-21:00")
    return s, e_min


def book(rooms, bookings, request):
    """Проверяет заявку и добавляет бронь при успехе.

    request = (room, day, start, end, who, people)

    Raises:
        RoomNotFoundError: аудитория отсутствует.
        InvalidTimeError: время некорректно.
        CapacityError: участников больше вместимости.
        TimeConflictError: пересечение с существующей бронью.
    """
    room, day, start, end, who, people = request
    if room not in rooms:
        raise RoomNotFoundError(room)
    s, e = check_time(start, end)
    capacity = rooms[room]
    if people > capacity:
        raise CapacityError(room, capacity, people)
    for existing in bookings:
        if existing[0] != room or existing[1] != day:
            continue
        es, ee = parse_time(existing[2]), parse_time(existing[3])
        if s < ee and es < e:
            raise TimeConflictError(room, (day, start, end), existing)
    bookings.append((room, day, start, end, who))


# ---------- разбор заявок и вывод ----------

def parse_request(line):
    parts = [p.strip() for p in line.rstrip("\n").split(";")]
    room, day, start, end, who, people = parts
    return (room, day, start, end, who, int(people))


def make_requests_file(name):
    data = (
        "412; пн; 10:00; 11:30; ЗФИ-01; 25\n"
        "415; пн; 12:00; 13:30; ЗФИ-02; 28\n"
        "412; пн; 11:00; 12:30; ЗФИ-03; 20\n"
        "301; вт; 09:00; 10:30; ЗФИ-04; 15\n"
        "501; ср; 10:00; 11:30; ЗФИ-05; 10\n"
        "412; пн; 14:00; 13:00; ЗФИ-06; 20\n"
        "204; чт; 08:00; 09:30; ЗФИ-07; 60\n"
        "412; пт; 19:00; 22:00; ЗФИ-08; 20\n"
        "415; сб; 09:00; 10:00; ЗФИ-09; 5\n"
        "301; вс; 10:00; 11:00; ЗФИ-10; 20\n"
    )
    with open(name, "w", encoding="utf-8") as f:
        f.write(data)


def process_requests(name):
    """Обрабатывает файл заявок, выводит результат по каждой.

    Returns:
        Список успешно добавленных броней.
    """
    rooms = {"412": 30, "415": 30, "301": 20, "204": 50, "501": 25}
    bookings = []
    with open(name, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            if not line.strip():
                continue
            request = parse_request(line)
            room, day, start, end, who, people = request
            try:
                book(rooms, bookings, request)
            except BookingError as e:
                print(f"Заявка {i}: {room}, {day} {start}-{end} – "
                      f"{type(e).__name__}: {e}")
            else:
                print(f"Заявка {i}: {room}, {day} {start}-{end}, "
                      f"группа {who} – забронировано")

    by_room = defaultdict(list)
    for b in bookings:
        by_room[b[0]].append(b)
    print("\nРасписание:")
    for room in sorted(by_room):
        print(f"  Аудитория {room}:")
        for b in sorted(by_room[room],
                        key=lambda x: (DAY_ORDER.get(x[1], 99),
                                       parse_time(x[2]))):
            print(f"    {b[1]} {b[2]}-{b[3]}, группа {b[4]}")
    return bookings


def main():
    filename = "requests.txt"
    make_requests_file(filename)
    process_requests(filename)


main()

Заявка 1: 412, пн 10:00-11:30, группа ЗФИ-01 – забронировано
Заявка 2: 415, пн 12:00-13:30, группа ЗФИ-02 – забронировано
Заявка 3: 412, пн 11:00-12:30 – TimeConflictError: занято группой ЗФИ-01 (10:00-11:30)
Заявка 4: 301, вт 09:00-10:30, группа ЗФИ-04 – забронировано
Заявка 5: 501, ср 10:00-11:30, группа ЗФИ-05 – забронировано
Заявка 6: 412, пн 14:00-13:00 – InvalidTimeError: начало позже окончания
Заявка 7: 204, чт 08:00-09:30 – CapacityError: аудитория 204: вместимость 50, заявлено 60
Заявка 8: 412, пт 19:00-22:00 – InvalidTimeError: вне пределов 08:00-21:00
Заявка 9: 415, сб 09:00-10:00, группа ЗФИ-09 – забронировано
Заявка 10: 301, вс 10:00-11:00, группа ЗФИ-10 – забронировано

Расписание:
  Аудитория 301:
    вт 09:00-10:30, группа ЗФИ-04
    вс 10:00-11:00, группа ЗФИ-10
  Аудитория 412:
    пн 10:00-11:30, группа ЗФИ-01
  Аудитория 415:
    пн 12:00-13:30, группа ЗФИ-02
    сб 09:00-10:00, группа ЗФИ-09
  Аудитория 501:
    ср 10:00-11:30, группа ЗФИ-05


In [9]:
import glob
import os
"""Deletes all .txt files in the current directory."""
for file in glob.glob("*.txt"):
    try:
        os.remove(file)
        print(f"Deleted: {file}")
    except OSError as e:
        print(f"Error deleting {file}: {e}")


for file in glob.glob("*.ini"):
    try:
        os.remove(file)
        print(f"Deleted: {file}")
    except OSError as e:
        
        print(f"Error deleting {file}: {e}")

Deleted: forms.txt
Deleted: operations.txt
Deleted: settings_1.ini
Deleted: settings_2.ini
Deleted: settings_3.ini
Deleted: settings_4.ini
Deleted: settings_ok.ini
